### File names
USREC.csv
2026-02-MD_edited.csv
2026-02-QD_edited.csv
credit_risk_dataset.csv

In [ ]:
# ============================================
# BASIC PREPROCESSING + EDA TEMPLATE
# Limited to at most 15 random columns for EDA
# Based only on the items in your guide
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# 1. LOAD DATA
# -----------------------------
# Replace with your file path
df = pd.read_csv("your_file.csv")

# Make a working copy
data = df.copy()

print("Initial shape:", data.shape)
display(data.head())


# -----------------------------
# 2. PREPROCESSING
# -----------------------------

print("")
print("PREPROCESSING")
print("")

# 2.1 Identify ID columns and remove them from modeling-style analysis
# Edit this list if needed
id_cols = [col for col in data.columns if "id" in col.lower()]

print("ID columns found:", id_cols)

# Keep IDs if you want for reference, but remove from analysis dataset
data_no_id = data.drop(columns=id_cols, errors="ignore").copy()


# 2.2 Handle missing values (drop NA, impute)
# Numeric -> median imputation
# Categorical -> mode imputation
for col in data_no_id.columns:
    if data_no_id[col].isna().sum() > 0:
        if pd.api.types.is_numeric_dtype(data_no_id[col]):
            data_no_id[col] = data_no_id[col].fillna(data_no_id[col].median())
        else:
            if data_no_id[col].mode().empty:
                data_no_id[col] = data_no_id[col].fillna("Missing")
            else:
                data_no_id[col] = data_no_id[col].fillna(data_no_id[col].mode()[0])

print("\nMissing values after handling:")
print(data_no_id.isna().sum().sort_values(ascending=False))


# 2.3 Convert categorical variables into binary flags (one-hot encoding)
categorical_cols = data_no_id.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
data_encoded = pd.get_dummies(data_no_id, columns=categorical_cols, drop_first=False)

print("\nShape after one-hot encoding:", data_encoded.shape)


# 2.4 Remove duplicates
before_dups = data_encoded.shape[0]
data_encoded = data_encoded.drop_duplicates()
after_dups = data_encoded.shape[0]

print("\nDuplicate rows removed:", before_dups - after_dups)
print("Shape after duplicate removal:", data_encoded.shape)


# 2.5 Detect outliers using IQR rule
numeric_cols = data_encoded.select_dtypes(include=[np.number]).columns.tolist()

outlier_summary = []

for col in numeric_cols:
    q1 = data_encoded[col].quantile(0.25)
    q3 = data_encoded[col].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_count = ((data_encoded[col] < lower_bound) | (data_encoded[col] > upper_bound)).sum()

    outlier_summary.append({
        "variable": col,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": outlier_count
    })

outlier_df = pd.DataFrame(outlier_summary).sort_values("outlier_count", ascending=False)

print("\nOutlier detection summary:")
display(outlier_df)


# 2.6 Filter out unrealistic values
# Fill this dictionary only for variables where you know valid limits.
# Example:
# realistic_ranges = {
#     "age": (0, 100),
#     "income": (0, 1000000)
# }

realistic_ranges = {}

data_clean = data_encoded.copy()

for col, (min_val, max_val) in realistic_ranges.items():
    if col in data_clean.columns:
        data_clean = data_clean[(data_clean[col] >= min_val) & (data_clean[col] <= max_val)]

print("\nShape after filtering unrealistic values:", data_clean.shape)


# -----------------------------
# 3. LIMIT EDA TO MAX 15 RANDOM COLUMNS
# -----------------------------
eda_data = data_no_id.copy()

max_eda_cols = 15
np.random.seed(42)

if eda_data.shape[1] > max_eda_cols:
    sampled_cols = np.random.choice(eda_data.columns, size=max_eda_cols, replace=False).tolist()
    eda_data = eda_data[sampled_cols].copy()
    print(f"\nEDA limited to 15 random columns out of {data_no_id.shape[1]} total columns.")
    print("Selected columns:")
    print(sampled_cols)
else:
    print(f"\nEDA uses all {eda_data.shape[1]} columns because total columns <= 15.")
    print("Selected columns:")
    print(eda_data.columns.tolist())


# -----------------------------
# 4. UNIVARIATE ANALYSIS
# -----------------------------

print("")
print("UNIVARIATE ANALYSIS")
print("")

# 4.1 Histograms of features
numeric_cols_eda = eda_data.select_dtypes(include=[np.number]).columns.tolist()

for col in numeric_cols_eda:
    plt.figure(figsize=(6, 4))
    plt.hist(eda_data[col].dropna(), bins=30)
    plt.title(f"Histogram: {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.show()


# 4.2 Bar charts of categorical variables
categorical_cols_eda = eda_data.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

for col in categorical_cols_eda:
    plt.figure(figsize=(6, 4))
    eda_data[col].value_counts(dropna=False).plot(kind="bar")
    plt.title(f"Bar Chart: {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.xticks(rotation=45)
    plt.show()


# 4.3 Summary statistics
print("\nSummary statistics for numeric variables:")
if len(numeric_cols_eda) > 0:
    display(eda_data[numeric_cols_eda].describe())
else:
    print("No numeric variables in sampled EDA columns.")

if len(categorical_cols_eda) > 0:
    print("\nSummary statistics for categorical variables:")
    display(eda_data[categorical_cols_eda].describe())


# 4.4 Identify skewed or bimodal distributions
distribution_notes = []

for col in numeric_cols_eda:
    skew_value = eda_data[col].skew()
    distribution_notes.append({
        "variable": col,
        "skewness": skew_value
    })

distribution_notes_df = pd.DataFrame(distribution_notes)

print("\nSkewness check:")
if not distribution_notes_df.empty:
    distribution_notes_df = distribution_notes_df.sort_values("skewness", key=np.abs, ascending=False)
    display(distribution_notes_df)
else:
    print("No numeric variables available for skewness check.")

print("\nBimodality check:")
print("Look at the histograms above to visually identify possible bimodal distributions.")


# 4.5 Detect extreme values
extreme_values = []

for col in numeric_cols_eda:
    extreme_values.append({
        "variable": col,
        "min": eda_data[col].min(),
        "max": eda_data[col].max()
    })

extreme_values_df = pd.DataFrame(extreme_values)

print("\nExtreme values summary:")
if not extreme_values_df.empty:
    display(extreme_values_df)
else:
    print("No numeric variables available for extreme value summary.")


# -----------------------------
# 5. BIVARIATE ANALYSIS
# -----------------------------

print("")
print("BIVARIATE ANALYSIS")
print("")

# Replace with your actual outcome variable name
outcome_col = "your_outcome_column"

# Only use outcome if it exists in the original non-ID dataset
outcome_available = outcome_col in data_no_id.columns

# 5.1 Scatterplots
# Scatterplots for numeric variable pairs among sampled columns
for i in range(len(numeric_cols_eda)):
    for j in range(i + 1, len(numeric_cols_eda)):
        x_col = numeric_cols_eda[i]
        y_col = numeric_cols_eda[j]

        plt.figure(figsize=(6, 4))
        plt.scatter(eda_data[x_col], eda_data[y_col], alpha=0.6)
        plt.title(f"Scatterplot: {x_col} vs {y_col}")
        plt.xlabel(x_col)
        plt.ylabel(y_col)
        plt.show()


# 5.2 Correlation matrix
if len(numeric_cols_eda) > 0:
    corr_matrix = eda_data[numeric_cols_eda].corr()

    plt.figure(figsize=(10, 8))
    plt.imshow(corr_matrix, aspect="auto")
    plt.colorbar()
    plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=90)
    plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)
    plt.title("Correlation Matrix")
    plt.show()

    print("\nCorrelation matrix:")
    display(corr_matrix)


# 5.3 Boxplots of outcomes by category
# Use sampled categorical columns only, but outcome from original data
if outcome_available:
    boxplot_data = data_no_id.copy()

    # keep only sampled categorical columns that still exist + outcome
    selected_cat_cols = [col for col in categorical_cols_eda if col in boxplot_data.columns]
    needed_cols = selected_cat_cols + [outcome_col]
    boxplot_data = boxplot_data[needed_cols].copy()

    for col in selected_cat_cols:
        plt.figure(figsize=(8, 4))
        boxplot_data.boxplot(column=outcome_col, by=col)
        plt.title(f"{outcome_col} by {col}")
        plt.suptitle("")
        plt.xlabel(col)
        plt.ylabel(outcome_col)
        plt.xticks(rotation=45)
        plt.show()
else:
    print(f"\nOutcome column '{outcome_col}' not found. Skipping boxplots, crosstabs, and grouped means.")


# 5.4 Crosstab tables
if outcome_available:
    for col in categorical_cols_eda:
        if col in data_no_id.columns:
            ctab = pd.crosstab(data_no_id[col], data_no_id[outcome_col], dropna=False)
            print(f"\nCrosstab: {col} vs {outcome_col}")
            display(ctab)


# 5.5 Grouped means
if outcome_available:
    for col in categorical_cols_eda:
        if col in data_no_id.columns:
            grouped_mean = data_no_id.groupby(col)[outcome_col].mean().reset_index()
            print(f"\nGrouped mean of {outcome_col} by {col}")
            display(grouped_mean)


# -----------------------------
# 6. FINAL DATA CHECK
# -----------------------------
print("\nFinal cleaned dataset shape:", data_clean.shape)
display(data_clean.head())

FileNotFoundError: [Errno 2] No such file or directory: 'credit_risk_dataset.csv'